# Assignment 1: Advanced Python and Statistical Analysis for Data Analytics

Name: Harshita Rai
Roll No.: 2503201000556 
Branch: CSE  
Section: 22  
College:ABES Engineering College

## Objective
This notebook implements all tasks from Assignment 1 using the given 40-student dataset. It covers Python functions, comprehensions, lambda expressions, CSV handling, Pandas, NumPy, descriptive statistics, correlation, confidence intervals, and hypothesis testing.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n = 40

df = pd.DataFrame({
    "Student_ID": range(1, n + 1),
    "Branch": np.random.choice(["CSE", "ECE", "ME", "CE"], n),
    "Study_Hours": np.round(np.random.uniform(1, 8, n), 1),
    "Attendance": np.random.randint(55, 100, n),
})

df["Marks"] = np.clip(
    np.round(30 + 5 * df["Study_Hours"] + 0.2 * df["Attendance"]
             + np.random.normal(0, 6, n)),
    0, 100
).astype(int)

df.head()

,Student_ID,Branch,Study_Hours,Attendance,Marks
0,1,ME,5.3,70,75
1,2,CE,2.0,99,68
2,3,CSE,3.0,72,59
3,4,ME,3.6,78,70
4,5,ME,4.2,80,69


Part A: Python for Data Analytics
Task 1: Grading Function

In [3]:
def get_grade(marks):
    if marks >= 75:
        return "A"
    elif marks >= 60:
        return "B"
    elif marks >= 40:
        return "C"
    else:
        return "F"

df["Grade"] = df["Marks"].apply(get_grade)

print("Grade counts:")
print(df["Grade"].value_counts())

Grade counts:
Grade
B    16
A    12
C    12
Name: count, dtype: int64


 Task 2: Comprehensions and Lambda

In [4]:
low_attendance_marks = [
    marks for marks, attendance in zip(df["Marks"], df["Attendance"])
    if attendance < 75
]

above_70 = list(filter(lambda x: x > 70, df["Marks"]))

grace_marks = list(map(lambda x: min(x + 5, 100), df["Marks"]))

print("Marks with attendance below 75:")
print(low_attendance_marks)

print("\nMarks above 70:")
print(above_70)

print("\nMarks after adding 5 grace marks:")
print(grace_marks)

Marks with attendance below 75:
[75, 59, 60, 57, 78, 82, 55, 80, 48, 64, 54, 58, 65, 82, 80, 69]

Marks above 70:
[75, 74, 81, 72, 78, 82, 80, 87, 78, 78, 83, 82, 80, 84]

Marks after adding 5 grace marks:
[80, 73, 64, 75, 74, 79, 69, 86, 77, 65, 66, 62, 59, 83, 87, 72, 60, 65, 85, 68, 53, 69, 59, 92, 64, 83, 63, 83, 73, 64, 88, 70, 87, 85, 74, 89, 52, 60, 55, 65]


Task 3: Flexible Functions

In [5]:
def summary(*values, round_to=2):
    return {
        "count": len(values),
        "minimum": min(values),
        "maximum": max(values),
        "average": round(np.mean(values), round_to)
    }

print("summary(10, 20, 30):")
print(summary(10, 20, 30))

print("\nsummary(*df['Marks']):")
print(summary(*df["Marks"]))

summary(10, 20, 30):
{'count': 3, 'minimum': 10, 'maximum': 30, 'average': np.float64(20.0)}

summary(*df['Marks']):
{'count': 40, 'minimum': 47, 'maximum': 87, 'average': np.float64(66.92)}


Task 4: Safe File Handling

In [7]:
df.to_csv("students.csv", index=False)

def load_data(filename):
    try:
        data = pd.read_csv(filename)
        print("File loaded successfully!")
        return data
    except FileNotFoundError:
        print("Sorry, file was not found.")
        return None

print("Correct filename:")
data1 = load_data("students.csv")

print("\nWrong filename:")
data2 = load_data("wrong_file.csv")

Correct filename:
File loaded successfully!

Wrong filename:
Sorry, file was not found.


Task 5: Grouping with Pandas

In [13]:
branch_avg = df.groupby("Branch")[["Marks", "Study_Hours"]].mean()

print("Average Marks and Study Hours by Branch:")
print(branch_avg)

highest_branch = branch_avg["Marks"].idxmax()
print("\nBranch with highest average marks:", highest_branch)

print("\nTop 5 students by marks:")
print(df.sort_values("Marks", ascending=False).head(5))

Average Marks and Study Hours by Branch:
            Marks  Study_Hours
Branch                        
CE      68.416667       4.4500
CSE     67.200000       4.6600
ECE     65.625000       3.7875
ME      65.900000       4.2200

Branch with highest average marks: CE

Top 5 students by marks:
    Student_ID Branch  Study_Hours  Attendance  Marks Grade
23          24     CE          7.4          89     87     A
35          36    CSE          7.5          88     84     A
30          31    CSE          7.8          82     83     A
14          15     CE          7.8          63     82     A
32          33     CE          7.6          63     82     A


Task 6: Fast Calculations with NumPy

In [12]:
marks_array = df["Marks"].to_numpy()

z_scores = (marks_array - marks_array.mean()) / marks_array.std()

print("Z-scores:")
print(z_scores)

print("\nNumber of students with z-score > 1:",
      np.sum(z_scores > 1))

Z-scores:
[ 0.74346049  0.09897462 -0.72965008  0.28311344  0.19104403  0.65139108
 -0.26930302  1.29587695  0.46725226 -0.63758067 -0.54551126 -0.9137889
 -1.18999713  1.01966872  1.38794636  0.00690521 -1.09792772 -0.63758067
  1.20380754 -0.36137243 -1.74241359 -0.26930302 -1.18999713  1.84829341
 -0.72965008  1.01966872 -0.82171949  1.01966872  0.09897462 -0.72965008
  1.48001577 -0.17723361  1.38794636  1.20380754  0.19104403  1.57208518
 -1.834483   -1.09792772 -1.55827477 -0.63758067]

Number of students with z-score > 1: 11


Part B: Statistical Analysis and Inference

Task 7: Describing the Marks

In [14]:
marks = df["Marks"]

print("Mean:", marks.mean())
print("Median:", marks.median())
print("Mode:", marks.mode().tolist())
print("Variance:", marks.var())
print("Standard Deviation:", marks.std())
print("Minimum:", marks.min())
print("Maximum:", marks.max())

Q1 = marks.quantile(0.25)
Q3 = marks.quantile(0.75)
IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

outliers = marks[(marks < lower_limit) | (marks > upper_limit)]
print("Outliers:", outliers.tolist())

Mean: 66.925
Median: 66.0
Mode: [59, 60, 78]
Variance: 120.99423076923075
Standard Deviation: 10.99973775911184
Minimum: 47
Maximum: 87
Q1: 59.0
Q3: 78.0
IQR: 19.0
Lower limit: 30.5
Upper limit: 106.5
Outliers: []


Task 8: Correlation

In [24]:
study_corr = df["Marks"].corr(df["Study_Hours"])
attendance_corr = df["Marks"].corr(df["Attendance"])

print("Correlation: Marks vs Study Hours =", study_corr)
print("Correlation: Marks vs Attendance =", attendance_corr)

Correlation: Marks vs Study Hours = 0.8537360674373369
Correlation: Marks vs Attendance = 0.06674385703072847


Answer:The relationship between Marks and Study Hours is stronger and positive. Attendance has a much weaker relationship with Marks in this dataset.

Why correlation does not prove causation: Correlation shows that two variables are related, but it does not prove that one variable directly causes the other.

Task 9: 95% Confidence Interval

In [23]:
confidence_interval = stats.t.interval(
    0.95,
    len(marks) - 1,
    loc=marks.mean(),
    scale=stats.sem(marks)
)

print("95% Confidence Interval:", confidence_interval)

95% Confidence Interval: (np.float64(63.407113198283106), np.float64(70.4428868017169))


**Interpretation:** We are 95% confident that the population average marks are represented by the calculated interval around the sample mean.

Task 10: Hypothesis Test on One Group

In [18]:
# H0: The average marks are 60.
# H1: The average marks are not 60.

t_stat, p_value = stats.ttest_1samp(df["Marks"], 60)

print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Decision: Reject H0")
    print("Conclusion: The average marks are significantly different from 60.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is not enough evidence that the average marks differ from 60.")

t-statistic: 3.981689977749835
p-value: 0.00028925643020879
Decision: Reject H0
Conclusion: The average marks are significantly different from 60.


Task 11: Hypothesis Test on Two Groups

In [22]:
cse_marks = df[df["Branch"] == "CSE"]["Marks"]
ece_marks = df[df["Branch"] == "ECE"]["Marks"]

# H0: CSE and ECE have equal average marks.
# H1: CSE and ECE have different average marks.

t_stat, p_value = stats.ttest_ind(cse_marks, ece_marks)

print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Decision: Reject H0")
    print("Conclusion: The average marks of CSE and ECE are significantly different.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is no statistically significant difference between CSE and ECE marks.")

t-statistic: 0.2757747137287823
p-value: 0.7862511358871336
Decision: Fail to reject H0
Conclusion: There is no statistically significant difference between CSE and ECE marks.


Part C: Summary — Task 12

The analysis describes the marks of 40 students using descriptive statistics and statistical tests. The marks are centered around the mid-60s, with moderate variation. Study Hours show a much stronger positive correlation with Marks than Attendance. The one-sample t-test checks the department's claim of an average of 60, while the two-sample test compares CSE and ECE. Overall, Study Hours appear to be the factor most strongly associated with Marks in this dataset.

Submission Checklist

- Name: Harshita rai
- Roll No.: 2503201000556
- Branch: CSE
- Section: 22
- All Tasks 1–12 included